In [1]:
from datetime import datetime


class RoomScheduler:
    def __init__(self):
        self._bookings_by_room = {}
        self._used_ids = set()

    def book(self, booking_id, room, start, end):
        # Basic input validation
        if not booking_id or type(booking_id) != str:
            raise ValueError("Booking ID valid string honi chahiye")
        if not room or type(room) != str:
            raise ValueError("Room naam valid string hona chahiye")

        if type(start) != datetime or type(end) != datetime:
            raise TypeError("start aur end datetime objects hone chahiye")

        if end <= start:
            raise ValueError("End datetime, start datetime se baad honi chahiye")

        if booking_id in self._used_ids:
            raise ValueError(f"Duplicate booking ID: '{booking_id}' pehle se maujood hai")

        # Room ke existing bookings check karein
        room_bookings = self._bookings_by_room.get(room, [])
        conflicts = []

        for existing in room_bookings:
            # Overlap condition (adjacent bookings allowed hain: start == existing.end ya end == existing.start)
            if start < existing["end"] and end > existing["start"]:
                conflicts.append(dict(existing))

        # Agar koi conflict mila to reject karein aur conflicting bookings return karein
        if len(conflicts) > 0:
            return {
                "accepted": False,
                "booking_id": booking_id,
                "room": room,
                "conflicts": conflicts
            }

        # Booking accept karein aur start time ke hisaab se sorted rakhein
        new_booking = {
            "booking_id": booking_id,
            "room": room,
            "start": start,
            "end": end
        }

        if room not in self._bookings_by_room:
            self._bookings_by_room[room] = []

        self._bookings_by_room[room].append(new_booking)
        self._bookings_by_room[room].sort(key=lambda item: item["start"])
        self._used_ids.add(booking_id)

        return {
            "accepted": True,
            "booking_id": booking_id,
            "room": room,
            "conflicts": []
        }

    def get_schedule(self, room):
        return [dict(b) for b in self._bookings_by_room.get(room, [])]


# Testing
if __name__ == "__main__":
    scheduler = RoomScheduler()

    t1_start = datetime(2026, 11, 7, 9, 0)
    t1_end = datetime(2026, 11, 7, 10, 0)

    t2_start = datetime(2026, 11, 7, 10, 0)
    t2_end = datetime(2026, 11, 7, 11, 0)

    t3_start = datetime(2026, 11, 7, 9, 30)
    t3_end = datetime(2026, 11, 7, 10, 30)

    print("Booking 1 (09:00-10:00):", scheduler.book("B1", "Room A", t1_start, t1_end))
    print("Booking 2 (10:00-11:00):", scheduler.book("B2", "Room A", t2_start, t2_end))
    print("Booking 3 (09:30-10:30):", scheduler.book("B3", "Room A", t3_start, t3_end))

Booking 1 (09:00-10:00): {'accepted': True, 'booking_id': 'B1', 'room': 'Room A', 'conflicts': []}
Booking 2 (10:00-11:00): {'accepted': True, 'booking_id': 'B2', 'room': 'Room A', 'conflicts': []}
Booking 3 (09:30-10:30): {'accepted': False, 'booking_id': 'B3', 'room': 'Room A', 'conflicts': [{'booking_id': 'B1', 'room': 'Room A', 'start': datetime.datetime(2026, 11, 7, 9, 0), 'end': datetime.datetime(2026, 11, 7, 10, 0)}, {'booking_id': 'B2', 'room': 'Room A', 'start': datetime.datetime(2026, 11, 7, 10, 0), 'end': datetime.datetime(2026, 11, 7, 11, 0)}]}
